# Benchmark models on the hard query sets (Colab)

In [ ]:
!nvidia-smi

## 1. Mount Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
%cd /content
if not os.path.isdir('mars-clip-bench'):
    !git clone https://github.com/anhkos/mars-clip-bench.git
%cd /content/mars-clip-bench
!git pull

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements-colab.txt

## 3. Link data / checkpoints / eval runs to Drive

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/mars-clip-bench-data"
for sub in ("data", "checkpoints", "eval_runs", "queries"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

!rm -rf data checkpoints eval/runs
!ln -s {DRIVE_ROOT}/data data
!ln -s {DRIVE_ROOT}/checkpoints checkpoints
!ln -s {DRIVE_ROOT}/eval_runs eval/runs

!python scripts/download_data.py --dataset all
!python finetune/make_hirise_splits.py

## 4. Point at your hard query files and validate them
Edit the two paths if your files live elsewhere. This cell checks the columns and class ids
so a bad file fails here instead of ten minutes into a run.

In [ ]:
import pandas as pd

QUERIES = {
    "msl":    f"{DRIVE_ROOT}/queries/msl_natural_queries.csv",
    "hirise": f"{DRIVE_ROOT}/queries/hirise_natural_queries.csv",
}
TAG = "natural"   # goes into output filenames so results never overwrite the default-query runs

for ds, path in QUERIES.items():
    df = pd.read_csv(path)
    missing = {"class_id", "class_name", "query_text"} - set(df.columns)
    assert not missing, f"{path} is missing columns: {missing}"
    print(f"{ds}: {len(df)} queries, {df.class_id.nunique()} classes")
    display(df.groupby(['class_id', 'class_name']).size().rename('n_queries').to_frame().T)
    display(df.sample(3, random_state=0)[['class_name', 'query_text']])

## 5. Choose models
Each entry becomes one eval run per dataset. Delete or comment out what you don't need.
Fine-tuned entries need the `.pt` files in `checkpoints/` on Drive (produced by the fine-tune notebook).
The RemoteCLIP/GeoRSCLIP entries need the download in the next cell.

In [ ]:
!python scripts/download_remote_sensing_checkpoints.py

In [ ]:
# (label, extra CLI args, datasets to run on)
MODELS = [
    ("clip_zeroshot",   "--model clip --model-id ViT-B/32",                                                                  ["msl", "hirise"]),
    ("clip_ft_msl",     "--model clip --model-id ViT-B/32 --checkpoint checkpoints/clip_finetuned.pt",                      ["msl"]),
    ("clip_ft_hirise",  "--model clip --model-id ViT-B/32 --checkpoint checkpoints/hirise_clip_finetuned.pt",               ["hirise"]),
    # ("openclip_b32",    "--model open_clip --model-id ViT-B-32 --pretrained laion2b_s34b_b79k",                              ["msl", "hirise"]),
    # ("siglip2_base",    "--model siglip --model-id google/siglip2-base-patch16-224",                                         ["msl", "hirise"]),
    # ("remoteclip",      "--model open_clip --model-id ViT-B-32 --pretrained none --checkpoint checkpoints/remoteclip_vit_b32.pt", ["msl", "hirise"]),
    # ("georsclip",       "--model open_clip --model-id ViT-B-32 --pretrained none --checkpoint checkpoints/georsclip_vit_b32.pt",  ["msl", "hirise"]),
]

## 6. Run the benchmarks
Image embeddings are cached on Drive per model, so reruns (or adding new queries) only re-encode the text.
A failing model is reported and skipped instead of stopping the whole sweep.

In [ ]:
import subprocess, shlex, time

failed = []
for label, args, datasets in MODELS:
    for ds in datasets:
        out = f"eval/runs/{label}_{ds}_{TAG}_results.csv"
        cmd = (f"python eval/model_agnostic_retrieval_eval.py {args} --dataset {ds} "
               f"--queries {shlex.quote(QUERIES[ds])} --out {out}")
        print(f"\n{'='*80}\n{label} on {ds}\n{'='*80}")
        t0 = time.time()
        rc = subprocess.run(cmd, shell=True).returncode
        if rc != 0:
            failed.append((label, ds))
            print(f"!! FAILED ({label}, {ds}) rc={rc}")
        else:
            print(f"-- done in {time.time()-t0:.0f}s")

print("\nFailed runs:", failed or "none")

## 7. Comparison table
Macro-averaged by class (same as the script's SUMMARY), from the per-query CSVs.

In [ ]:
import glob, re

cols = ["p@1", "p@5", "p@10", "r@10", "map", "ndcg@10"]
rows = []
for path in sorted(glob.glob(f"eval/runs/*_{TAG}_results.csv")):
    m = re.match(rf".*/(.+)_(msl|hirise)_{TAG}_results\.csv", path)
    label, ds = m.group(1), m.group(2)
    df = pd.read_csv(path)
    macro = df.groupby("class_label")[[c for c in cols if c in df.columns]].mean().mean()
    rows.append({"model": label, "dataset": ds, "n_queries": len(df), **macro.round(4).to_dict()})

summary = pd.DataFrame(rows)
for ds in ("msl", "hirise"):
    print(f"\n=== {ds.upper()} (hard queries) ===")
    display(summary[summary.dataset == ds].drop(columns="dataset").sort_values("ndcg@10", ascending=False).reset_index(drop=True))

summary.to_csv(f"eval/runs/summary_{TAG}.csv", index=False)

## 8. Hardest queries
Per-query failures for one model — useful for the writeup. Change `LABEL` / `DS`.

In [ ]:
LABEL, DS = "clip_zeroshot", "msl"
df = pd.read_csv(f"eval/runs/{LABEL}_{DS}_{TAG}_results.csv")
display(df.sort_values("ndcg@10").head(15)[["class_label", "query_text", "p@1", "p@10", "ndcg@10"]])

Everything is saved on Drive: per-query CSVs and `summary_hard.csv` in `MyDrive/mars-clip-bench-data/eval_runs/`.